# Lab 2 tasks
Name: Khair ul wara Hussain

Student Id: FA23-BAI-022

Course: NLP


In [7]:
# ==============================================================================
# TASK 1: Multi-Page PDF Extraction with Word Frequency Analysis
# ==============================================================================
print("\n" + "="*80)
print("TASK 1")
print("="*80)

pdf_path = "sample_doc.pdf"  # Update with your local PDF filename

# Task 1(a): Extract page text
pages = []
with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    for p in reader.pages:
        txt = p.extract_text()
        pages.append(txt if txt else "")

print("Total pages read:", len(pages))

# Task 1(b): Merge, clean punctuation/digits, tokenize
full_doc = " ".join(pages)
clean_doc = re.sub(r'[^a-zA-Z\s]', '', full_doc).lower()
all_tokens = clean_doc.split()

print("Total words found:", len(all_tokens))

# Task 1(c): Remove stopwords and count top 10 words
sw = set(stopwords.words('english'))
filtered_words = [word for word in all_tokens if word not in sw and len(word) > 1]

freq_map = Counter(filtered_words)
print("\nTop 10 words:")
for word, count in freq_map.most_common(10):
    print(f"{word}: {count}")

# Task 1(d): Keyword sentence search across pages
query = input("\nEnter keyword to search: ").strip()
found_any = False

for idx, page_content in enumerate(pages, start=1):
    sentences = re.split(r'[.!?]+', page_content)
    for s in sentences:
        if re.search(rf'\b{re.escape(query)}\b', s, re.IGNORECASE):
            s_clean = " ".join(s.split())
            print(f"[Page {idx}] {s_clean}")
            found_any = True

if not found_any:
    print(f"No sentences found with keyword: '{query}'")


TASK 1
Total pages read: 1
Total words found: 607

Top 10 words:
data: 7
built: 7
fine: 6
rag: 5
ai: 5
accuracy: 5
python: 4
pipelines: 4
llm: 4
pipeline: 4

Enter keyword to search: Khair ul wara
[Page 1] KHAIR UL WARA HUSSAIN AI/ML Engineer | Python Developer Islamabad, Pakistan | +92 314 1839730 | khairulwarahussain251203@gmail


In [8]:
# ==============================================================================
# TASK 2: Structured Page Metadata Extraction and CSV Export
# ==============================================================================
print("\n" + "="*80)
print("TASK 2")
print("="*80)

class PhraseNotFoundError(Exception):
    pass

# Task 2(a): Build metadata list
pages_data = []

with open(pdf_path, "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)
    for p_num, page_obj in enumerate(pdf_reader.pages, 1):
        txt = page_obj.extract_text() or ""
        w_count = len(txt.split())
        c_count = len(re.sub(r'\s+', '', txt)) # exclude spaces

        pages_data.append({
            "page_number": p_num,
            "raw_text": txt,
            "word_count": w_count,
            "char_count": c_count
        })

print("First two pages metadata:")
pprint(pages_data[:2])

# Task 2(b): Stats on word counts
max_page = max(pages_data, key=lambda x: x["word_count"])
valid_pages = [p for p in pages_data if p["word_count"] > 0]
min_page = min(valid_pages, key=lambda x: x["word_count"]) if valid_pages else None

avg_words = sum(p["word_count"] for p in pages_data) / len(pages_data) if pages_data else 0

print(f"\nPage with max words: Page {max_page['page_number']} ({max_page['word_count']} words)")
print("Sample content:", max_page['raw_text'][:150])

if min_page:
    print(f"Page with min words (>0): Page {min_page['page_number']} ({min_page['word_count']} words)")

print(f"Average words per page: {round(avg_words, 2)}")

# Task 2(c): Custom Exception phrase search
target_phrase = input("\nEnter phrase to search: ").strip()

try:
    matches_found = False
    for item in pages_data:
        counts = len(re.findall(re.escape(target_phrase), item["raw_text"], re.IGNORECASE))
        if counts > 0:
            print(f"Page {item['page_number']}: '{target_phrase}' appeared {counts} times.")
            matches_found = True

    if not matches_found:
        raise PhraseNotFoundError(f"Phrase '{target_phrase}' was not found in the file.")

except PhraseNotFoundError as err:
    print("Error caught:", err)

# Task 2(d): CSV export and read back
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as out_f:
    writer = csv.writer(out_f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for row in pages_data:
        writer.writerow([row["page_number"], row["word_count"], row["char_count"]])

print("\nVerifying written CSV contents:")
with open("pdf_summary.csv", "r", encoding="utf-8") as in_f:
    reader = csv.reader(in_f)
    for r in reader:
        print(r)


TASK 2
First two pages metadata:
[{'char_count': 4129,
  'page_number': 1,
  'raw_text': 'KHAIR UL WARA HUSSAIN  \n'
              'AI/ML Engineer | Python Developer  \n'
              'Islamabad, Pakistan | +92 314 1839730 | '
              'khairulwarahussain251203@gmail.com  \n'
              'linkedin.com/in/khair -ul-wara -hussain -629452283  | '
              'github.com/khairulwarahussain251203 -max \n'
              'PROFILE  \n'
              'AI/ML and full -stack engineer with hands -on experience '
              'building NLP, RAG, and computer -vision systems, automated data '
              'pipelines, and production MLOps \n'
              'infrastructure. Skilled at translating research and messy data '
              'into deployable tools, dashboards, and appl ications, with a '
              'track record of taking projects end -to-\n'
              'end from research to working product.  \n'
              'EDUCATION  \n'
              'BS in Artificial Intelligence —

In [9]:
# ==============================================================================
# TASK 3: PDF Text Preprocessing Pipeline for NLP
# ==============================================================================
print("\n" + "="*80)
print("TASK 3")
print("="*80)

# Task 3(a): Regex Cleaning
all_text = " ".join([p["raw_text"] for p in pages_data])
orig_len = len(all_text)

t_no_url = re.sub(r'https?://\S+|www\.\S+', '', all_text)
t_no_num = re.sub(r'\d+', '', t_no_url)
t_alpha = re.sub(r'[^a-zA-Z\s\.]', '', t_no_num)
t_clean = re.sub(r'\s+', ' ', t_alpha)

print(f"Characters before clean: {orig_len}")
print(f"Characters after clean:  {len(t_clean)}")

# Task 3(b): Sentence Tokenization
lowercase_text = t_clean.lower()
raw_sents = re.split(r'[.!?]', lowercase_text)

stripped_sents = [s.strip() for s in raw_sents]
valid_sents = [s for s in stripped_sents if len(s) > 0]

print(f"Sentences count (raw): {len(raw_sents)}")
print(f"Sentences count (cleaned): {len(valid_sents)}")

# Task 3(c): Stopwords and short sentence filter
stop_words = set(stopwords.words('english'))
processed_sents = []

for sentence in valid_sents:
    tokens = sentence.split()
    tokens_no_sw = [w for w in tokens if w not in stop_words]
    if len(tokens_no_sw) >= 5:
        processed_sents.append(" ".join(tokens_no_sw))

print(f"Sentences count before 5-token filter: {len(valid_sents)}")
print(f"Sentences count after 5-token filter:  {len(processed_sents)}")

print("\nSample 3 processed sentences:")
for line in processed_sents[:3]:
    print("-", line)

# Task 3(d): Save output to text file and read back
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for line in processed_sents:
        f.write(line + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    file_lines = [l.strip() for l in f if l.strip()]

longest = max(file_lines, key=len) if file_lines else ""
shortest = min(file_lines, key=len) if file_lines else ""

print(f"\nLines written to file: {len(file_lines)}")
print(f"Longest sentence ({len(longest)} chars): {longest}")
print(f"Shortest sentence ({len(shortest)} chars): {shortest}")


TASK 3
Characters before clean: 4943
Characters after clean:  4365
Sentences count (raw): 23
Sentences count (cleaned): 23
Sentences count before 5-token filter: 23
Sentences count after 5-token filter:  21

Sample 3 processed sentences:
- khair ul wara hussain aiml engineer python developer islamabad pakistan khairulwarahussaingmail
- comkhairulwarahussain max profile aiml full stack engineer hands experience building nlp rag computer vision systems automated data pipelines production mlops infrastructure
- skilled translating research messy data deployable tools dashboards appl ications track record taking projects end end research working product

Lines written to file: 21
Longest sentence (853 chars): technical skills ai ml scikit learn langchain bert bart rag fine tuning cnns random forest isolation forest time series forecasting ensemble methods data science pandas numpy matplotlib eda feature engineering geospatial analysis statistical modeling backend fastapi rest apis sqlalch

In [12]:
# ==============================================================================
# TASK 4: DOCX Paragraph Classification and Statistical Analysis
# ==============================================================================
print("\n" + "="*80)
print("TASK 4")
print("="*80)

docx_file = "sample_doc.docx"  # Update with your local .docx filename

# Check if the DOCX file exists before proceeding with DOCX-related tasks
if not os.path.exists(docx_file):
    raise FileNotFoundError(f"The file '{docx_file}' was not found. Please upload 'sample_doc.docx' to your Colab environment or update the 'docx_file' variable to point to an existing DOCX file.")

document = docx.Document(docx_file)

# Task 4(a): Extract paragraph and style
para_list = []
for p in document.paragraphs:
    para_list.append({
        "text": p.text,
        "style": p.style.name
    })

unique_styles = set(item["style"] for item in para_list)
print("Total paragraphs:", len(para_list))
print("Unique styles found:", list(unique_styles))

# Task 4(b): Categorize paragraphs
heading_paras = []
body_paras = []
other_paras = []

for item in para_list:
    text = item["text"].strip()
    st = item["style"]

    if st.startswith("Heading"):
        heading_paras.append(item)
    elif (st == "Normal" or "Body" in st) and text != "":
        body_paras.append(item)
    else:
        other_paras.append(item)

print(f"\nHeadings count: {len(heading_paras)}")
print(f"Body count:     {len(body_paras)}")
print(f"Other count:    {len(other_paras)}")

# Task 4(c): Analyze body paragraphs (> 5 words)
filtered_body = []
for b in body_paras:
    words = b["text"].split()
    if len(words) > 5:
        ch_count = len(re.sub(r'\s+', '', b["text"]))
        filtered_body.append({
            "text": b["text"],
            "words": len(words),
            "chars": ch_count
        })

if filtered_body:
    max_b = max(filtered_body, key=lambda x: x["words"])
    min_b = min(filtered_body, key=lambda x: x["words"])
    mean_words = sum(b["words"] for b in filtered_body) / len(filtered_body)

    print("\nBody Stats:")
    print(f"Max word count paragraph ({max_b['words']} words):\n{max_b['text'][:120]}...")
    print(f"Min word count paragraph ({min_b['words']} words):\n{min_b['text']}")
    print(f"Mean word count: {round(mean_words, 2)}")

# Task 4(d): Nested style map
style_dict = {}
for p in para_list:
    st_name = p["style"]
    if st_name not in style_dict:
        style_dict[st_name] = {"count": 0, "texts": []}
    style_dict[st_name]["count"] += 1
    style_dict[st_name]["texts"].append(p["text"])

for s_name in style_dict:
    if len(style_dict[s_name]["texts"]) > 2:
        style_dict[s_name]["texts"] = style_dict[s_name]["texts"][:2]

sorted_styles = dict(sorted(style_dict.items(), key=lambda item: item[1]["count"], reverse=True))

print("\nStyles Dictionary Output:")
pprint(sorted_styles)


TASK 4
Total paragraphs: 14
Unique styles found: ['Normal', 'List Paragraph']

Headings count: 0
Body count:     8
Other count:    6

Styles Dictionary Output:
{'List Paragraph': {'count': 2,
                    'texts': ['The student-intern is to fill this form by '
                              'recording major tasks performed by them.',
                              'Respective internship supervisors are to review '
                              'and evaluate the student’s activity log.']},
 'Normal': {'count': 12,
            'texts': ['COMSATS University Islamabad', 'Islamabad Campus']}}


In [13]:
# ==============================================================================
# TASK 5: DOCX Table Extraction and Pandas Integration
# ==============================================================================
print("\n" + "="*80)
print("TASK 5")
print("="*80)

# Task 5(a): Extract tables into raw lists
extracted_tables = []

for table in document.tables:
    table_content = []
    for row in table.rows:
        row_vals = [cell.text.strip() for cell in row.cells]
        table_content.append(row_vals)
    extracted_tables.append(table_content)

print("Tables count:", len(extracted_tables))
for i, tbl in enumerate(extracted_tables, 1):
    r_count = len(tbl)
    c_count = len(tbl[0]) if r_count > 0 else 0
    print(f"Table {i}: {r_count} rows x {c_count} cols")

# Task 5(b): Parse first table into DataFrame
if extracted_tables:
    first_tbl = extracted_tables[0]
    headers = first_tbl[0]
    data = first_tbl[1:]

    df = pd.DataFrame(data, columns=headers)
    print("\nTable DataFrame:")
    print(df)

    empty_cells = df.applymap(lambda x: x == "" or x is None).sum()
    print("\nEmpty cells count per column:")
    print(empty_cells)

# Task 5(c): Extract paragraphs prior to first table
pre_paras = []
for el in document.element.body:
    if el.tag.endswith('tbl'):
        break
    if el.tag.endswith('p'):
        p_text = docx.text.paragraph.Paragraph(el, document).text.strip()
        if p_text:
            pre_paras.append(p_text)

print(f"\nParagraphs before the first table: {len(pre_paras)}")

# Task 5(d): Output combined text file
with open("docx_extracted.txt", "w", encoding="utf-8") as out:
    out.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for p in pre_paras:
        out.write(p + "\n")

    out.write("\n--- TABLE DATA ---\n")
    if extracted_tables:
        for row in extracted_tables[0]:
            out.write(" | ".join(row) + "\n")

print("Saved data to docx_extracted.txt successfully.")


TASK 5
Tables count: 2
Table 1: 2 rows x 2 cols
Table 2: 7 rows x 3 cols

Table DataFrame:
        Intern Name: Khair ul wara Hussain  \
0  Faculty Supervisor: Mr. Taimorr Shahzad   

                  Organization: AKSA-SDS  
0  Reporting Period: 06/07/26 – 17/08/26  

Empty cells count per column:
Intern Name: Khair ul wara Hussain    0
Organization: AKSA-SDS                0
dtype: int64

Paragraphs before the first table: 4
Saved data to docx_extracted.txt successfully.


/tmp/ipykernel_3173/3564191311.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  empty_cells = df.applymap(lambda x: x == "" or x is None).sum()


In [16]:
# ==============================================================================
# TASK 6: Multi-Document Comparison, Deduplication, and Frequency Report
# ==============================================================================
print("\n" + "="*80)
print("TASK 6")
print("="*80)

doc1_file = "sample_doc.docx"
doc2_file = "sample_doc_2.docx"

if not os.path.exists(doc1_file):
    raise FileNotFoundError(f"The file '{doc1_file}' was not found. Please upload 'sample_doc.docx' to your Colab environment or update the variable to point to an existing DOCX file.")

if not os.path.exists(doc2_file):
    raise FileNotFoundError(f"The file '{doc2_file}' was not found. Please upload 'sample_doc_2.docx' to your Colab environment or update the variable to point to an existing DOCX file.")

doc1 = docx.Document(doc1_file)   # Document A
doc2 = docx.Document(doc2_file) # Document B

# Task 6(a): Set analysis on Headings
h_a = [p.text.strip() for p in doc1.paragraphs if p.style.name.startswith("Heading") and p.text.strip()]
h_b = [p.text.strip() for p in doc2.paragraphs if p.style.name.startswith("Heading") and p.text.strip()]

set_head_a = set(h_a)
set_head_b = set(h_b)

print("Common headings:", set_head_a.intersection(set_head_b))
print("Only in Doc A:", set_head_a - set_head_b)
print("Only in Doc B:", set_head_b - set_head_a)

# Task 6(b): Merge Normal paragraphs & deduplicate
b_a = [p.text.strip() for p in doc1.paragraphs if p.style.name == "Normal" and p.text.strip()]
b_b = [p.text.strip() for p in doc2.paragraphs if p.style.name == "Normal" and p.text.strip()]

all_body = b_a + b_b
seen_paras = set()
unique_body = []

for p in all_body:
    normalized_p = " ".join(p.lower().split())
    if normalized_p not in seen_paras:
        seen_paras.add(normalized_p)
        unique_body.append(p)

print(f"\nTotal normal paras before dedup: {len(all_body)}")
print(f"Total normal paras after dedup:  {len(unique_body)}")

# Task 6(c): Word counting across body text
doc_sw = set(stopwords.words('english'))
words_list = []

for p in unique_body:
    clean_p = re.sub(r'[^a-zA-Z\s]', '', p).lower()
    tokens = [w for w in clean_p.split() if w not in doc_sw and len(w) > 1]
    words_list.extend(tokens)

word_counts = Counter(words_list)
print("\nTop 20 Frequent Words:")
for word, count in word_counts.most_common(20):
    print(f"{word}: {count}")

# Task 6(d): Summary report
total_heads = len(h_a) + len(h_b)
vocab_size = len(word_counts)
total_words = sum(len(p.split()) for p in unique_body)
avg_len = total_words / len(unique_body) if unique_body else 0
longest_para = max(unique_body, key=lambda x: len(x.split())) if unique_body else ""

print("\n--- Summary ---")
print("1. Total Headings:", total_heads)
print("2. Unique Body Paragraphs:", len(unique_body))
print("3. Vocabulary Size:", vocab_size)
print("4. Average Paragraph Length:", round(avg_len, 2), "words")
print(f"5. Longest Paragraph ({len(longest_para.split())} words):\n{longest_para}")


TASK 6
Common headings: set()
Only in Doc A: set()
Only in Doc B: {'Summer 2026', 'Student Name: Khair ul wara Hussain', 'Program: AI', 'Students’ Career Goal', 'Submitted By', 'Internship Placement Overview & Role', 'Scope of Work', 'Internship Design Proposal', 'Supervisor Details', 'Learning Objectives', 'Submission Date: 9th September 2026', 'Expected Deliverables:', 'Academic Preparation', 'COMSATS University Islamabad, Islamabad Campus', 'Key Activities (check all that apply):', 'Tools/Technologies to be used:', 'Registration No: FA23-BAI-022'}

Total normal paras before dedup: 10
Total normal paras after dedup:  10

Top 20 Frequent Words:
signature: 3
islamabad: 2
supervisor: 2
engineering: 2
model: 2
pipeline: 2
google: 2
api: 2
comsats: 1
university: 1
campus: 1
department: 1
computer: 1
science: 1
student: 1
internship: 1
activity: 1
log: 1
instruction: 1
studentintern: 1

--- Summary ---
1. Total Headings: 17
2. Unique Body Paragraphs: 10
3. Vocabulary Size: 80
4. Average P

In [17]:
# ==============================================================================
# TASK 7: Multi-Endpoint API Data Joining and Analysis
# ==============================================================================
print("\n" + "="*80)
print("TASK 7")
print("="*80)

# Task 7(a): Fetch datasets
res_users = requests.get("https://jsonplaceholder.typicode.com/users")
if res_users.status_code != 200:
    raise ConnectionError(f"Failed to fetch users. Code: {res_users.status_code}")

res_posts = requests.get("https://jsonplaceholder.typicode.com/posts")
if res_posts.status_code != 200:
    raise ConnectionError(f"Failed to fetch posts. Code: {res_posts.status_code}")

users_json = res_users.json()
posts_json = res_posts.json()

# Task 7(b): Join datasets
merged_users = {}

for u in users_json:
    uid = u["id"]
    uname = u["name"]
    user_titles = [p["title"] for p in posts_json if p["userId"] == uid]

    merged_users[uname] = {
        "email": u["email"],
        "company": u["company"]["name"],
        "post_count": len(user_titles),
        "titles": user_titles
    }

print("Joined Data (First 3 users):")
first_3 = dict(list(merged_users.items())[:3])
pprint(first_3)

# Task 7(c): User with most posts
top_user = max(merged_users.items(), key=lambda item: item[1]["post_count"])
name = top_user[0]
info = top_user[1]

print(f"\nUser with most posts: {name}")
print(f"Email: {info['email']}")
print(f"Company: {info['company']}")
print("Posts:")
for idx, title in enumerate(info["titles"], 1):
    print(f" {idx}. {title}")

# Task 7(d): Filter users
print("\nFiltered Users (synergies in BS OR city length > 8):")
for u in users_json:
    bs = u.get("company", {}).get("bs", "").lower()
    city = u.get("address", {}).get("city", "")

    if "synergies" in bs or len(city) > 8:
        p_count = sum(1 for p in posts_json if p["userId"] == u["id"])
        print(f"Name: {u['name']} | City: {city} | Posts: {p_count}")


TASK 7
Joined Data (First 3 users):
{'Clementine Bauch': {'company': 'Romaguera-Jacobson',
                      'email': 'Nathan@yesenia.net',
                      'post_count': 10,
                      'titles': ['asperiores ea ipsam voluptatibus modi minima '
                                 'quia sint',
                                 'dolor sint quo a velit explicabo quia nam',
                                 'maxime id vitae nihil numquam',
                                 'autem hic labore sunt dolores incidunt',
                                 'rem alias distinctio quo quis',
                                 'est et quae odit qui non',
                                 'quasi id et eos tenetur aut quo autem',
                                 'delectus ullam et corporis nulla voluptas '
                                 'sequi',
                                 'iusto eius quod necessitatibus culpa ea',
                                 'a quo magni similique perferendis']},


In [18]:
# ==============================================================================
# TASK 8: To-Do Completion Rate Analysis Across Users
# ==============================================================================
print("\n" + "="*80)
print("TASK 8")
print("="*80)

# Task 8(a): Fetch and validate
res_todos = requests.get("https://jsonplaceholder.typicode.com/todos")

if res_todos.status_code != 200 or not isinstance(res_todos.json(), list) or len(res_todos.json()) == 0:
    print("Invalid response or missing data.")
    exit()

todos_list = res_todos.json()
print("Retrieved todos count:", len(todos_list))

# Task 8(b): Group per user
user_todos = {}

for t in todos_list:
    uid = t["userId"]
    if uid not in user_todos:
        user_todos[uid] = {"total": 0, "completed": 0, "incomplete": 0}

    user_todos[uid]["total"] += 1
    if t["completed"]:
        user_todos[uid]["completed"] += 1
    else:
        user_todos[uid]["incomplete"] += 1

print("\nValidation of task totals per user:")
for uid, stats in user_todos.items():
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats} | Total matches: {check}")

# Task 8(c): Completion rate metrics
rates = {}
for uid, stats in user_todos.items():
    pct = (stats["completed"] / stats["total"]) * 100
    rates[uid] = round(pct, 1)

best_user = max(rates.items(), key=lambda x: x[1])
worst_user = min(rates.items(), key=lambda x: x[1])
avg_completion = sum(rates.values()) / len(rates)

print(f"\nBest completion rate: User {best_user[0]} ({best_user[1]}%)")
print(f"Worst completion rate: User {worst_user[0]} ({worst_user[1]}%)")
print(f"Average rate across users: {round(avg_completion, 1)}%")

# Task 8(d): Leaderboard table
raw_users_data = requests.get("https://jsonplaceholder.typicode.com/users").json()
id_to_name = {u["id"]: u["name"] for u in raw_users_data}

table_data = []
for uid, stats in user_todos.items():
    table_data.append({
        "id": uid,
        "name": id_to_name.get(uid, f"User {uid}"),
        "total": stats["total"],
        "completed": stats["completed"],
        "rate": rates[uid]
    })

table_data.sort(key=lambda x: x["rate"], reverse=True)

print("\n" + "-"*65)
print(f"{'Rank':<6} | {'Name':<22} | {'Total':<8} | {'Done':<8} | {'Rate %':<8}")
print("-" * 65)
for rank, row in enumerate(table_data, 1):
    print(f"{rank:<6} | {row['name']:<22} | {row['total']:<8} | {row['completed']:<8} | {row['rate']:<8.1f}")
print("-" * 65)


TASK 8
Retrieved todos count: 200

Validation of task totals per user:
User 1: {'total': 20, 'completed': 11, 'incomplete': 9} | Total matches: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12} | Total matches: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13} | Total matches: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14} | Total matches: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8} | Total matches: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14} | Total matches: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11} | Total matches: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9} | Total matches: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12} | Total matches: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8} | Total matches: True

Best completion rate: User 5 (60.0%)
Worst completion rate: User 4 (30.0%)
Average rate across users: 45.0%

-------------------------------------

In [19]:
# ==============================================================================
# TASK 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis
# ==============================================================================
print("\n" + "="*80)
print("TASK 9")
print("="*80)

# Task 9(a): Flattener function
def flatten_user(u):
    return {
        "id": u.get("id"),
        "name": u.get("name"),
        "username": u.get("username"),
        "email": u.get("email"),
        "city": u.get("address", {}).get("city"),
        "zipcode": u.get("address", {}).get("zipcode"),
        "lat": u.get("address", {}).get("geo", {}).get("lat"),
        "lng": u.get("address", {}).get("geo", {}).get("lng"),
        "company_name": u.get("company", {}).get("name"),
        "company_catchphrase": u.get("company", {}).get("catchPhrase")
    }

users_api = requests.get("https://jsonplaceholder.typicode.com/users").json()
flat_users = [flatten_user(u) for u in users_api]

print("Sample flattened record:")
pprint(flat_users[0])

# Task 9(b): DataFrame & Distance calculation
df_u = pd.DataFrame(flat_users)

df_u["lat"] = df_u["lat"].astype(float)
df_u["lng"] = df_u["lng"].astype(float)

df_u["distance_from_origin"] = df_u.apply(
    lambda r: math.sqrt(r["lat"]**2 + r["lng"]**2), axis=1
)

farthest_user = df_u.loc[df_u["distance_from_origin"].idxmax()]
print(f"\nFarthest user: {farthest_user['name']} (Dist: {round(farthest_user['distance_from_origin'], 2)})")

# Task 9(c): Group by Hemisphere
north = df_u[df_u["lat"] > 0]
south = df_u[df_u["lat"] < 0]

print("\nNorthern Hemisphere users:", north["name"].tolist())
print("Southern Hemisphere users:", south["name"].tolist())

avg_n = north["distance_from_origin"].mean()
avg_s = south["distance_from_origin"].mean()

print(f"Avg Distance (North): {round(avg_n, 2)}")
print(f"Avg Distance (South): {round(avg_s, 2)}")

if avg_n > avg_s:
    print("Northern hemisphere has a larger average distance.")
else:
    print("Southern hemisphere has a larger average distance.")

# Task 9(d): Posts and comments counter
posts_res = requests.get("https://jsonplaceholder.typicode.com/posts").json()
comments_res = requests.get("https://jsonplaceholder.typicode.com/comments").json()

comment_counter = Counter(c["postId"] for c in comments_res)

posts_with_counts = []
for p in posts_res:
    p_id = p["id"]
    short_title = p["title"][:40]
    count = comment_counter[p_id]
    posts_with_counts.append((p_id, short_title, count))

posts_with_counts.sort(key=lambda item: item[2], reverse=True)

print("\nTop 5 Most Commented Posts:")
for p in posts_with_counts[:5]:
    print(f"ID {p[0]:<3} | Title: {p[1]:<40} | Comments: {p[2]}")

print("\nBottom 5 Least Commented Posts:")
for p in posts_with_counts[-5:]:
    print(f"ID {p[0]:<3} | Title: {p[1]:<40} | Comments: {p[2]}")


TASK 9
Sample flattened record:
{'city': 'Gwenborough',
 'company_catchphrase': 'Multi-layered client-server neural-net',
 'company_name': 'Romaguera-Crona',
 'email': 'Sincere@april.biz',
 'id': 1,
 'lat': '-37.3159',
 'lng': '81.1496',
 'name': 'Leanne Graham',
 'username': 'Bret',
 'zipcode': '92998-3874'}

Farthest user: Glenna Reichert (Dist: 170.68)

Northern Hemisphere users: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere users: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']
Avg Distance (North): 123.58
Avg Distance (South): 84.32
Northern hemisphere has a larger average distance.

Top 5 Most Commented Posts:
ID 1   | Title: sunt aut facere repellat provident occae | Comments: 5
ID 2   | Title: qui est esse                             | Comments: 5
ID 3   | Title: ea molestias quasi exercitationem repell | Comments: 5
ID 4   | Title: eum et est oc